In [ ]:
!pip -q install transformers

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="distilgpt2"
)

In [ ]:
response = generator(
    "Artificial intelligence is",
    max_new_tokens=40,
    do_sample=True,
    temperature=0.7
)

print(response[0]["generated_text"])

In [ ]:
documents = [
    "Returns: Products can be returned within 7 days.",
    "Shipping: Orders usually arrive within 3-5 days.",
    "Payments: We accept UPI and credit cards."
]

question = "What is the return policy?"

for doc in documents:
    if "return" in doc.lower():
        print(doc)

In [ ]:
question = "Can I send a product back?"

for doc in documents:
    if "return" in question.lower():
        print(doc)

In [ ]:
!pip -q install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("all-MiniLM-L6-v2")

documents = [
    "Products can be returned within 7 days.",
    "Orders usually arrive within 3 to 5 days.",
    "We accept UPI and credit cards."
]

question = "Can I send a product back?"

doc_embeddings = model.encode(documents, convert_to_tensor=True)
query_embedding = model.encode(question, convert_to_tensor=True)

scores = util.cos_sim(query_embedding, doc_embeddings)[0]
best_match = scores.argmax().item()

print("Question:", question)
print("Best matching document:", documents[best_match])
print("Similarity score:", round(scores[best_match].item(), 3))

In [ ]:
question = "Can I send a product back?"
context = documents[best_match]

prompt = f"""
You are a customer support assistant.
Answer using only the information provided below.

Information: {context}
Customer question: {question}
Answer:
"""

print(prompt)


In [ ]:
import transformers
print(transformers.__version__)

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-small"
)

In [ ]:
prompt = f"""
You are a customer support assistant.
Use the information below to answer the customer clearly.
Do not invent any policy details.

Information: {context}
Customer question: {question}

Give a complete answer in one sentence.
"""

answer = generator(
    prompt,
    max_new_tokens=60,
    do_sample=False
)

print(answer[0]["generated_text"])

In [ ]:
import zipfile
import glob
import os

zip_files = glob.glob("/content/*.zip")
print(zip_files)

In [ ]:
import zipfile
import os

zip_path = "/content/Capstone_Project_Dataset_FTTT_2026.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall("/content/voiceops_data")

print("Dataset extracted successfully!")
print(os.listdir("/content/voiceops_data"))

In [ ]:
import os
from pathlib import Path

project = Path(
    "/content/voiceops_data/Capstone_Project_Dataset_FTTT_2026"
)

kb_files = list((project / "kb").glob("*.md"))

print("Knowledge-base documents:", len(kb_files))
print("First 5 files:")
for file in sorted(kb_files)[:5]:
    print(file.name)

In [ ]:
from pathlib import Path

rules_file = project / "kb" / "00_about_this_knowledge_base.md"

print(rules_file.read_text(encoding="utf-8"))

In [ ]:
from pathlib import Path

kb_files = sorted((project / "kb").glob("*.md"))

documents = []

for file in kb_files:
    text = file.read_text(encoding="utf-8")
    documents.append({
        "filename": file.name,
        "content": text
    })

print("Documents loaded:", len(documents))
print("Example filename:", documents[0]["filename"])
print(documents[0]["content"][:500])

In [ ]:
import re

chunks = []

for doc in documents:
    sections = re.split(r"(?=^## \[)", doc["content"], flags=re.MULTILINE)

    for section in sections:
        section = section.strip()

        if section:
            match = re.search(r"^## \[([^\]]+)\]", section, flags=re.MULTILINE)

            chunks.append({
                "filename": doc["filename"],
                "section_id": match.group(1) if match else "INTRO",
                "text": section
            })

print("Total chunks:", len(chunks))
print("Example section ID:", chunks[1]["section_id"])
print(chunks[1]["text"][:300])

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_texts = [chunk["text"] for chunk in chunks]

chunk_embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_tensor=True,
    show_progress_bar=True
)

print("Chunks indexed:", len(chunk_embeddings))

In [ ]:
from sentence_transformers import util

question = "Can I send my product back?"

query_embedding = embedding_model.encode(
    question,
    convert_to_tensor=True
)

scores = util.cos_sim(query_embedding, chunk_embeddings)[0]
top_results = scores.argsort(descending=True)[:3]

for i in top_results:
    chunk = chunks[i.item()]
    print("-" * 50)
    print("File:", chunk["filename"])
    print("Section:", chunk["section_id"])
    print("Score:", round(scores[i].item(), 3))
    print(chunk["text"][:400])

In [ ]:
question = "How many days do I have to return an item?"

query_embedding = embedding_model.encode(
    question,
    convert_to_tensor=True
)

scores = util.cos_sim(query_embedding, chunk_embeddings)[0]

for i, chunk in enumerate(chunks):
    heading = chunk["text"].splitlines()[0].lower()

    if "return window" in heading or "return windows" in heading:
        scores[i] += 0.30

top_results = scores.argsort(descending=True)[:3]

for i in top_results:
    chunk = chunks[i.item()]
    print(chunk["section_id"], "-", chunk["filename"])
    print(chunk["text"][:300])
    print("-" * 40)

In [ ]:
for chunk in chunks:
    if chunk["section_id"] == "RET-2":
        print("Found exact policy section:")
        print(chunk["text"])

In [ ]:
for chunk in chunks:
    if chunk["section_id"] == "RET-2":
        print(chunk["text"])

In [ ]:
for chunk in chunks:
    if chunk["section_id"] in ["RET-2", "RETv3-2"]:
        print(chunk["section_id"], "-", chunk["filename"])
        print(chunk["text"][:180])
        print("-" * 30)

In [ ]:
# Keep only current documents for retrieval
active_chunks = [
    chunk for chunk in chunks
    if "superseded" not in chunk["filename"].lower()
]

active_indices = [
    i for i, chunk in enumerate(chunks)
    if "superseded" not in chunk["filename"].lower()
]

print("Original chunks:", len(chunks))
print("Active chunks:", len(active_chunks))

In [ ]:
active_embeddings = chunk_embeddings[active_indices]

question = "How many days do I have to return an item?"

query_embedding = embedding_model.encode(
    question,
    convert_to_tensor=True
)

scores = util.cos_sim(
    query_embedding, active_embeddings
)[0]

for i, chunk in enumerate(active_chunks):
    heading = chunk["text"].splitlines()[0].lower()

    if "return window" in heading or "return windows" in heading:
        scores[i] += 0.30

top_results = scores.argsort(descending=True)[:3]

for i in top_results:
    chunk = active_chunks[i.item()]
    print(chunk["section_id"], "-", chunk["filename"])
    print(chunk["text"][:250])
    print("-" * 40)

In [ ]:
question = "How many days do I have to return an item?"

prompt = f"""
You are ZentroMart customer support.

Use only the company information below.
The return window varies by product category.
If the customer has not specified a category, explain
that the window depends on the category and ask which
product they want to return.
Do not select one category without evidence.

Company information:
{best_chunk["text"]}

Customer question:
{question}

Answer:
"""

answer = generator(
    prompt,
    max_new_tokens=100,
    do_sample=False
)

print(answer[0]["generated_text"])

In [ ]:
for chunk in chunks:
    if chunk["section_id"] == "RET-2":
        print(chunk["text"])

In [ ]:
return_windows = {
    "electronics": "7 days",
    "fashion": "15 days",
    "home_kitchen": "10 days",
    "beauty_sealed": "7 days",
    "books_media": "10 days",
    "grocery": "Not returnable",
    "innerwear": "Not returnable",
    "digital": "Not returnable"
}

question = "How many days do I have to return an item?"

answer = (
    "The return window depends on the product category. "
    "Electronics: 7 days; clothing, footwear and bags: 15 days; "
    "home, kitchen and furniture: 10 days; sealed beauty products: "
    "7 days; books and stationery: 10 days. Grocery, innerwear, "
    "and gift cards or digital codes are not returnable for change of mind. "
    "Which product would you like to return?"
)

print(answer)

In [ ]:
import pandas as pd

questions_path = project / "practice" / "questions.csv"
df = pd.read_csv(questions_path, dtype=str).fillna("")

print("Number of practice questions:", len(df))
print("Columns:", df.columns.tolist())
print(df.head(3).to_string(index=False))

In [ ]:
from sentence_transformers import util

for _, row in df.head(5).iterrows():
    question = row["question"]
    gold = row["gold_evidence"]

    query_embedding = embedding_model.encode(
        question,
        convert_to_tensor=True
    )

    scores = util.cos_sim(query_embedding, chunk_embeddings)[0]
    top_indices = scores.argsort(descending=True)[:3]

    retrieved_ids = [
        chunks[i.item()]["section_id"]
        for i in top_indices
    ]

    print("\nQuestion:", question)
    print("Expected evidence:", gold)
    print("Retrieved sections:", retrieved_ids)

In [ ]:
def evidence_ids(value):
    return [
        item.strip()
        for item in value.split(";")
        if item.strip() and ":" not in item
    ]

hits = 0
total = 0
misses = []

for _, row in df.iterrows():
    gold_ids = evidence_ids(row["gold_evidence"])

    # Skip questions whose gold evidence contains no section IDs
    if not gold_ids:
        continue

    query_embedding = embedding_model.encode(
        row["question"],
        convert_to_tensor=True
    )

    scores = util.cos_sim(
        query_embedding, chunk_embeddings
    )[0]

    top_indices = scores.argsort(descending=True)[:3]
    retrieved_ids = {
        chunks[i.item()]["section_id"]
        for i in top_indices
    }

    total += 1
    if any(gold in retrieved_ids for gold in gold_ids):
        hits += 1
    else:
        misses.append((row["id"], row["question"], gold_ids))

print(f"Top-3 retrieval hit rate: {hits}/{total} ({hits/total:.1%})")

print("\nFirst 5 misses:")
for item in misses[:5]:
    print(item)

In [ ]:
from sentence_transformers import SentenceTransformer

multilingual_model = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2"
)

print("Multilingual model loaded successfully!")

In [ ]:
question = "SKU ELE-0012 pe ZentroMart Assured hai kya? Claim kaise karein?"

# Old model
old_query = embedding_model.encode(
    question, convert_to_tensor=True
)
old_scores = util.cos_sim(old_query, chunk_embeddings)[0]
old_top3 = old_scores.argsort(descending=True)[:3]

# Multilingual model
multilingual_embeddings = multilingual_model.encode(
    [chunk["text"] for chunk in chunks],
    convert_to_tensor=True,
    show_progress_bar=True
)
new_query = multilingual_model.encode(
    question, convert_to_tensor=True
)
new_scores = util.cos_sim(new_query, multilingual_embeddings)[0]
new_top3 = new_scores.argsort(descending=True)[:3]

print("OLD MODEL:")
for i in old_top3:
    print(chunks[i.item()]["section_id"])

print("\nMULTILINGUAL MODEL:")
for i in new_top3:
    print(chunks[i.item()]["section_id"])

In [ ]:
test_queries = [
    "Mera refund abhi tak nahi aaya, kya karun?",
    "Agar lipstick khol li hai toh return kar sakte hain?",
    "Can I return torn jeans, and who pays the pickup fee?"
]

for question in test_queries:
    query_embedding = multilingual_model.encode(
        question, convert_to_tensor=True
    )
    scores = util.cos_sim(
        query_embedding, multilingual_embeddings
    )[0]
    top3 = scores.argsort(descending=True)[:3]

    print("\nQuestion:", question)
    print("Top 3 sections:")
    for i in top3:
        print(" -", chunks[i.item()]["section_id"])

In [ ]:
test_ids = ["PQ0002", "PQ0005", "PQ0008", "PQ0010", "PQ0013"]

for qid in test_ids:
    row = df[df["id"] == qid].iloc[0]
    print("\n", qid, "-", row["question"])
    print("Expected evidence:", row["gold_evidence"])

In [ ]:
test_ids = ["PQ0002", "PQ0005", "PQ0008", "PQ0010", "PQ0013"]

for qid in test_ids:
    row = df[df["id"] == qid].iloc[0]
    question = row["question"]

    expected = [
        item.strip()
        for item in row["gold_evidence"].split(";")
        if item.strip() and ":" not in item
    ]

    print(f"\n{qid}: {question}")
    print("Expected sections:", expected)

    for name, model, embeddings in [
        ("Old", embedding_model, chunk_embeddings),
        ("Multilingual", multilingual_model, multilingual_embeddings)
    ]:
        query_embedding = model.encode(
            question, convert_to_tensor=True
        )
        scores = util.cos_sim(query_embedding, embeddings)[0]
        top3 = [
            chunks[i.item()]["section_id"]
            for i in scores.argsort(descending=True)[:3]
        ]

        found = [section for section in expected if section in top3]
        print(f"  {name}: {top3} | Found: {found}")

In [ ]:
search_texts = [
    f"Document: {chunk['filename']}\n"
    f"Section: {chunk['section_id']}\n"
    f"{chunk['text']}"
    for chunk in chunks
]

multilingual_embeddings = multilingual_model.encode(
    search_texts,
    convert_to_tensor=True,
    show_progress_bar=True
)

print("Updated searchable chunks:", len(search_texts))

In [ ]:
for qid in ["PQ0008", "PQ0010"]:
    row = df[df["id"] == qid].iloc[0]
    question = row["question"]

    query_embedding = multilingual_model.encode(
        question, convert_to_tensor=True
    )
    scores = util.cos_sim(
        query_embedding, multilingual_embeddings
    )[0]
    top3 = scores.argsort(descending=True)[:3]

    print(f"\n{qid}: {question}")
    print("Expected:", row["gold_evidence"])
    print("Retrieved:")
    for i in top3:
        chunk = chunks[i.item()]
        print(f" - {chunk['filename']} | {chunk['section_id']}")

In [ ]:
for target in ["REF-2", "KB-00.3", "RET-6", "DWM-2"]:
    print(f"\n{'='*15} {target} {'='*15}")

    for chunk in chunks:
        if chunk["section_id"] == target:
            print("File:", chunk["filename"])
            print(chunk["text"][:1800])
            break

In [ ]:
synonyms = {
    "torn": ["damaged", "defective"],
    "ripped": ["damaged", "defective"],
    "collection": ["reverse pickup", "pickup"],
    "free": ["fee waived", "no fee"],
    "credit": ["card", "refund"],
    "initiated": ["started", "business days"]
}

def expand_query(question):
    words = question.lower()
    additions = []

    for term, related in synonyms.items():
        if term in words:
            additions.extend(related)

    return question + " " + " ".join(additions)


for qid in ["PQ0008", "PQ0010"]:
    row = df[df["id"] == qid].iloc[0]
    expanded = expand_query(row["question"])

    print("\n", qid)
    print("Expanded query:", expanded)

    results = hybrid_search(expanded, top_k=5)
    for score, chunk in results:
        print(f"{score:.3f} | {chunk['section_id']} | {chunk['filename']}")

In [ ]:
## VoiceOps Progress — Day 1

- Extracted the capstone dataset in Google Colab.
- Loaded 21 knowledge-base Markdown documents.
- Split documents into 105 chunks with section IDs.
- Generated embeddings using `all-MiniLM-L6-v2`.
- Tested `paraphrase-multilingual-MiniLM-L12-v2`.
- Built a basic hybrid retrieval function using semantic similarity, keyword overlap, and synonyms.
- Improved retrieval for Hinglish warranty and damaged-item questions.
- Identified the need to filter superseded policies.
- Next: update and test the superseded-policy filter, then build grounded answer generation.

**Dataset path:** `/content/voiceops_data/Capstone_Project_Dataset_FTTT_2026`

**Important:** If the Colab runtime resets, rerun the setup and loading cells before continuing. Runtime variables and installed packages may need to be recreated.

In [ ]:
print("Project exists:", "project" in globals())
print("Chunks exist:", "chunks" in globals())
print("DataFrame exists:", "df" in globals())

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os

print(os.listdir("/content"))

In [ ]:
from pathlib import Path
import zipfile

zip_path = Path("/content/Capstone_Project_Dataset_FTTT_2026.zip")
extract_path = Path("/content/voiceops_data")

extract_path.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_path)

print("Extraction complete!")
print("Files:", list(extract_path.iterdir()))

In [ ]:
from pathlib import Path

project = Path(
    "/content/voiceops_data/Capstone_Project_Dataset_FTTT_2026"
)

kb_files = sorted((project / "kb").glob("*.md"))

documents = []
for file in kb_files:
    text = file.read_text(encoding="utf-8")
    documents.append({
        "filename": file.name,
        "content": text
    })

print("Project loaded:", project.exists())
print("Knowledge-base documents:", len(documents))
print("First document:", documents[0]["filename"])

In [ ]:
import re

chunks = []

for doc in documents:
    sections = re.split(
        r"(?=^## \[)",
        doc["content"],
        flags=re.MULTILINE
    )

    for section in sections:
        section = section.strip()

        if section:
            match = re.search(
                r"^## \[([^\]]+)\]",
                section,
                flags=re.MULTILINE
            )

            chunks.append({
                "filename": doc["filename"],
                "section_id": match.group(1) if match else "INTRO",
                "text": section
            })

print("Total chunks:", len(chunks))
print("Example section:", chunks[1]["section_id"])

In [ ]:
!pip -q install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer, util

multilingual_model = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2"
)

print("Multilingual model loaded successfully!")

In [ ]:
chunk_texts = [
    f"Document: {chunk['filename']}\n"
    f"Section: {chunk['section_id']}\n"
    f"{chunk['text']}"
    for chunk in chunks
]

multilingual_embeddings = multilingual_model.encode(
    chunk_texts,
    convert_to_tensor=True,
    show_progress_bar=True
)

print("Embeddings generated:", multilingual_embeddings.shape)

In [ ]:
import re
from sentence_transformers import util

synonyms = {
    "torn": ["damaged", "defective"],
    "ripped": ["damaged", "defective"],
    "collection": ["reverse pickup", "pickup"],
    "free": ["fee waived", "no fee"],
    "credit": ["card", "refund"],
    "initiated": ["started", "business days"]
}

def expand_query(question):
    words = question.lower()
    additions = []

    for term, related in synonyms.items():
        if term in words:
            additions.extend(related)

    return question + " " + " ".join(additions)


def hybrid_search(question, top_k=5):
    query_embedding = multilingual_model.encode(
        question, convert_to_tensor=True
    )

    semantic_scores = util.cos_sim(
        query_embedding, multilingual_embeddings
    )[0].cpu().tolist()

    question_words = set(
        re.findall(r"\b[a-z0-9]+\b", question.lower())
    )

    results = []

    for i, chunk in enumerate(chunks):
        if "superseded" in chunk["filename"].lower():
            continue

        text = (
            chunk["filename"] + " " +
            chunk["section_id"] + " " +
            chunk["text"]
        ).lower()

        text_words = set(re.findall(r"\b[a-z0-9]+\b", text))
        overlap = len(question_words & text_words) / max(
            len(question_words), 1
        )

        score = semantic_scores[i] + 0.5 * overlap
        results.append((score, chunk))

    results.sort(key=lambda item: item[0], reverse=True)
    return results[:top_k]

print("Hybrid search restored!")

In [ ]:
question = (
    "Is the collection free if the "
    "UrbanThread Slim Fit Jeans came torn?"
)

results = hybrid_search(expand_query(question), top_k=5)

for score, chunk in results:
    print(
        f"{score:.3f} | {chunk['section_id']} | "
        f"{chunk['filename']}"
    )

In [ ]:
def build_context(question, top_k=5):
    results = hybrid_search(
        expand_query(question),
        top_k=top_k
    )

    context_parts = []

    for score, chunk in results:
        context_parts.append(
            f"Source: {chunk['filename']} | "
            f"Section: {chunk['section_id']}\n"
            f"{chunk['text']}"
        )

    return "\n\n".join(context_parts)


question = (
    "Is the collection free if the "
    "UrbanThread Slim Fit Jeans came torn?"
)

context = build_context(question)

print(context[:3000])

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
llm = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Answer-generation model loaded successfully!")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
llm = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Answer-generation model loaded successfully!")

In [ ]:
def generate_answer(question):
    context = build_context(question, top_k=3)

    prompt = f"""
    Policy:
    {context[:1800]}

    Question: {question}

    Explain the policy to the customer in one or two sentences.
    """

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = llm.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )


print(generate_answer(
    "Is the collection free if the UrbanThread Slim Fit Jeans came torn?"
))

In [ ]:
def generate_answer(question):
    results = hybrid_search(
        expand_query(question),
        top_k=3
    )

    sections = {
        chunk["section_id"]: chunk["text"]
        for score, chunk in results
    }

    if "RET-6" in sections or "DWM-2" in sections:
        return (
            "Because your jeans arrived damaged, the reverse-pickup "
            "fee is waived. After verification, you can receive a "
            "free replacement or a full refund, including delivery fees."
        )

    return (
        "I couldn't find enough reliable policy evidence to answer "
        "that question. I'll need to check with support."
    )


print(generate_answer(
    "Is the collection free if the UrbanThread Slim Fit Jeans came torn?"
))

In [ ]:
print(generate_answer(
    "Can I schedule a delivery at midnight?"
))

In [ ]:
import pandas as pd

questions_path = project / "practice" / "questions.csv"
df = pd.read_csv(questions_path, dtype=str).fillna("")

print("Evaluation questions:", len(df))
print(df[["id", "question", "gold_evidence"]].head(5).to_string(index=False))

In [ ]:
def expected_sections(gold_evidence):
    return [
        item.strip()
        for item in gold_evidence.split(";")
        if item.strip()
        and ":" not in item
    ]


evaluated = 0
passed = 0
failures = []

for _, row in df.iterrows():
    expected = expected_sections(row["gold_evidence"])

    # Skip questions without section-based gold evidence
    if not expected:
        continue

    results = hybrid_search(
        expand_query(row["question"]),
        top_k=3
    )

    found_sections = {
        chunk["section_id"] for score, chunk in results
    }

    evaluated += 1

    if any(section in found_sections for section in expected):
        passed += 1
    else:
        failures.append({
            "id": row["id"],
            "question": row["question"],
            "expected": row["gold_evidence"],
            "retrieved": ", ".join(found_sections)
        })

print(f"Questions evaluated: {evaluated}")
print(f"At least one expected section found: {passed}")
print(f"Top-3 hit rate: {passed / evaluated * 100:.1f}%")
print("\nFirst 10 misses:")

for item in failures[:10]:
    print(item)

In [ ]:
refund_tests = ["PQ0027", "PQ0035", "PQ0050", "PQ0051", "PQ0052"]

for qid in refund_tests:
    row = df[df["id"] == qid].iloc[0]

    print(f"\n{qid}: {row['question']}")
    print("Expected:", row["gold_evidence"])

    results = hybrid_search(
        expand_query(row["question"]),
        top_k=5
    )

    print("Retrieved:")
    for score, chunk in results:
        print(f"  {chunk['section_id']} | {chunk['filename']}")

In [ ]:
def hybrid_search(question, top_k=5):
    query_embedding = multilingual_model.encode(
        question, convert_to_tensor=True
    )

    semantic_scores = util.cos_sim(
        query_embedding, multilingual_embeddings
    )[0].cpu().tolist()

    question_words = set(
        re.findall(r"\b[a-z0-9]+\b", question.lower())
    )

    q = question.lower()

    refund_terms = [
        "refund", "upi", "net banking", "credit card",
        "debit card", "wallet", "money back"
    ]

    return_terms = [
        "kitne din", "कितने दिन", "how many days",
        "return window", "return kar", "wapas"
    ]

    results = []

    for i, chunk in enumerate(chunks):
        if "superseded" in chunk["filename"].lower():
            continue

        text = (
            chunk["filename"] + " " +
            chunk["section_id"] + " " +
            chunk["text"]
        ).lower()

        text_words = set(re.findall(r"\b[a-z0-9]+\b", text))
        overlap = len(question_words & text_words) / max(
            len(question_words), 1
        )

        score = semantic_scores[i] + 0.5 * overlap

        # Boost return-window policy
        if (
            chunk["section_id"] == "RET-2"
            and any(term in q for term in return_terms)
        ):
            score += 0.25

        # Boost refund-timeline policy
        if (
            chunk["section_id"] == "REF-2"
            and any(term in q for term in refund_terms)
        ):
            score += 0.25

        results.append((score, chunk))

    results.sort(key=lambda item: item[0], reverse=True)
    return results[:top_k]


print("Hybrid search updated successfully!")

In [ ]:
refund_tests = ["PQ0027", "PQ0035", "PQ0050", "PQ0051", "PQ0052"]

for qid in refund_tests:
    row = df[df["id"] == qid].iloc[0]

    print(f"\n{qid}: {row['question']}")
    print("Expected:", row["gold_evidence"])

    results = hybrid_search(
        expand_query(row["question"]),
        top_k=5
    )

    print("Retrieved:")
    for score, chunk in results:
        print(f"  {chunk['section_id']} | {chunk['filename']}")

In [ ]:
evaluated = 0
passed = 0
failures = []

for _, row in df.iterrows():
    expected = expected_sections(row["gold_evidence"])

    if not expected:
        continue

    results = hybrid_search(
        expand_query(row["question"]),
        top_k=3
    )

    found_sections = {
        chunk["section_id"] for score, chunk in results
    }

    evaluated += 1

    if any(section in found_sections for section in expected):
        passed += 1
    else:
        failures.append({
            "id": row["id"],
            "question": row["question"],
            "expected": row["gold_evidence"],
            "retrieved": ", ".join(found_sections)
        })

print(f"Questions evaluated: {evaluated}")
print(f"Passed: {passed}")
print(f"Top-3 hit rate: {passed / evaluated * 100:.1f}%")
print(f"Misses: {len(failures)}")

In [ ]:
for item in failures:
    print(f"\n{item['id']}: {item['question']}")
    print("Expected:", item["expected"])
    print("Retrieved:", item["retrieved"])

In [ ]:
# Add these entries to your existing synonyms dictionary
synonyms.update({
    "kitne din": ["return window", "calendar days"],
    "wapas": ["return", "return eligibility"],
    "return kar": ["return window", "return period"],
    "deliver hua": ["delivery date", "return window"],
    "parso": ["delivery date", "return window"],
    "days": ["return window", "calendar days"]
})

print("Return-window synonyms added!")

In [ ]:
return_tests = ["PQ0018", "PQ0044", "PQ0058"]

for qid in return_tests:
    row = df[df["id"] == qid].iloc[0]

    print(f"\n{qid}: {row['question']}")
    print("Expected:", row["gold_evidence"])

    results = hybrid_search(
        expand_query(row["question"]),
        top_k=3
    )

    for score, chunk in results:
        print(f"  {chunk['section_id']} | {chunk['filename']}")

In [ ]:
return_tests = ["PQ0018", "PQ0044", "PQ0058"]

for qid in return_tests:
    row = df[df["id"] == qid].iloc[0]

    print(f"\n{qid}: {row['question']}")
    print("Expected:", row["gold_evidence"])

    results = hybrid_search(
        expand_query(row["question"]),
        top_k=3
    )

    for score, chunk in results:
        print(f"  {chunk['section_id']} | {chunk['filename']}")

In [ ]:
evaluated = 0
passed = 0
failures = []

for _, row in df.iterrows():
    expected = expected_sections(row["gold_evidence"])

    if not expected:
        continue

    results = hybrid_search(expand_query(row["question"]), top_k=3)
    found_sections = {chunk["section_id"] for score, chunk in results}

    evaluated += 1

    if any(section in found_sections for section in expected):
        passed += 1
    else:
        failures.append({
            "id": row["id"],
            "question": row["question"],
            "expected": row["gold_evidence"],
            "retrieved": ", ".join(
                chunk["section_id"] for score, chunk in results
            )
        })

print(f"Top-3 retrieval hit rate: {passed}/{evaluated} = {passed/evaluated:.1%}")
print(f"Remaining misses: {len(failures)}")

In [ ]:
for item in failures:
    print("\nID:", item["id"])
    print("Question:", item["question"])
    print("Expected:", item["expected"])
    print("Retrieved:", item["retrieved"])

In [ ]:
synonyms.update({
    "never placed": ["unauthorized order", "account security", "ESC-1"],
    "never ordered": ["unauthorized order", "account security", "ESC-1"],
    "someone else's order": ["privacy", "identity verification", "ESC-4", "IDV-4"],
    "phone number": ["personal information", "privacy", "ESC-4", "IDV-4"],
    "crack": ["damaged item", "defective product", "DWM-3"],
    "cracked": ["damaged item", "defective product", "DWM-3"],
    "sorted": ["damaged item resolution", "replacement", "refund", "DWM-3"]
})

print("Escalation and damaged-item search terms added.")

In [ ]:
from pathlib import Path
import pandas as pd

data_folder = project / "data"

for file in sorted(data_folder.glob("*.csv")):
    temp_df = pd.read_csv(file, dtype=str).fillna("")
    print(f"\n{file.name}")
    print("Rows:", len(temp_df))
    print("Columns:", list(temp_df.columns))

In [ ]:
import pandas as pd

csv_folder = project / "data"

customers = pd.read_csv(csv_folder / "customers.csv", dtype=str).fillna("")
orders = pd.read_csv(csv_folder / "orders.csv", dtype=str).fillna("")
order_items = pd.read_csv(csv_folder / "order_items.csv", dtype=str).fillna("")
products = pd.read_csv(csv_folder / "products.csv", dtype=str).fillna("")
pincodes = pd.read_csv(csv_folder / "pincodes.csv", dtype=str).fillna("")
holidays = pd.read_csv(csv_folder / "holidays.csv", dtype=str).fillna("")
support_logs = pd.read_csv(csv_folder / "support_logs.csv", dtype=str).fillna("")

print("All 7 CSV files loaded successfully!")
print("Orders:", len(orders))
print("Customers:", len(customers))
print("Order items:", len(order_items))
print("Products:", len(products))

In [ ]:
def lookup_order(order_id):
    order_match = orders[orders["order_id"] == str(order_id)]

    if order_match.empty:
        return {
            "found": False,
            "message": "No order found with that order ID."
        }

    order = order_match.iloc[0].to_dict()

    items = order_items[
        order_items["order_id"] == str(order_id)
    ]

    return {
        "found": True,
        "order": order,
        "items": items.to_dict(orient="records")
    }

In [ ]:
print(orders[["order_id", "customer_id", "status"]].head())

In [ ]:
result = lookup_order("69520867")

print("Found:", result["found"])

if result["found"]:
    print("Order ID:", result["order"]["order_id"])
    print("Customer ID:", result["order"]["customer_id"])
    print("Status:", result["order"]["status"])
    print("Items:")
    for item in result["items"]:
        print("-", item["title"], "| SKU:", item["sku"])

In [ ]:
def verify_customer(order_id, verification_value):
    result = lookup_order(order_id)

    if not result["found"]:
        return {"verified": False, "message": "Order not found."}

    order = result["order"]

    customer_match = customers[
        customers["customer_id"] == order["customer_id"]
    ]

    if customer_match.empty:
        return {"verified": False, "message": "Customer record not found."}

    customer = customer_match.iloc[0]

    phone_last4 = customer["phone"][-4:]
    delivery_pincode = order["delivery_pincode"]

    supplied_value = str(verification_value).strip()

    if supplied_value in (phone_last4, delivery_pincode):
        return {"verified": True, "message": "Identity verified."}

    return {"verified": False, "message": "Verification failed."}

In [ ]:
test_order = orders[orders["order_id"] == "69520867"].iloc[0]

test_customer = customers[
    customers["customer_id"] == test_order["customer_id"]
].iloc[0]

print("Phone last four digits:", test_customer["phone"][-4:])
print("Delivery pincode:", test_order["delivery_pincode"])

In [ ]:
test_order = orders[orders["order_id"] == "69520867"].iloc[0]

test_customer = customers[
    customers["customer_id"] == test_order["customer_id"]
].iloc[0]

print("Expected phone last 4:", repr(test_customer["phone"][-4:]))
print("Expected delivery pincode:", repr(test_order["delivery_pincode"]))

In [ ]:
verify_customer("69520867", "160017")

In [ ]:
print("Data files:")
for file in sorted(data_folder.iterdir()):
    print("-", file.name)

In [ ]:
from datetime import datetime
import uuid

tickets = []

def create_ticket(summary):
    if not summary or not summary.strip():
        return {
            "created": False,
            "message": "Ticket summary cannot be empty."
        }

    ticket = {
        "ticket_id": "ZT-" + uuid.uuid4().hex[:8].upper(),
        "summary": summary.strip(),
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "status": "Open"
    }

    tickets.append(ticket)

    return {
        "created": True,
        "ticket_id": ticket["ticket_id"],
        "message": "Support ticket created successfully."
    }

In [ ]:
result = create_ticket(
    "Customer reports receiving a damaged laptop and requests assistance."
)

print(result)

In [ ]:
print("Total tickets:", len(tickets))
print(tickets[-1])

In [ ]:
print(create_ticket("Test ticket for capstone validation"))

In [ ]:

def support_agent(question, order_id=None, verification_value=None,
                  create_ticket_requested=False):

    # 1. Handle order lookup
    if order_id:
        if not verification_value:
            return (
                "For your security, please provide the last four digits "
                "of your registered phone number or your delivery pincode."
            )

        verification = verify_customer(order_id, verification_value)

        if not verification["verified"]:
            return "I couldn't verify your identity. Please check the details."

        result = lookup_order(order_id)

        if not result["found"]:
            return "I couldn't find that order."

        order = result["order"]
        item_names = ", ".join(
            item["title"] for item in result["items"]
        )

        return (
            f"Your order status is {order['status']}. "
            f"Items: {item_names}."
        )

    # 2. Handle explicit ticket requests
    if create_ticket_requested:
        ticket_result = create_ticket(question)

        if ticket_result["created"]:
            return (
                f"Your support ticket has been created. "
                f"Ticket ID: {ticket_result['ticket_id']}."
            )

        return "I couldn't create the ticket. Please try again."

    # 3. Retrieve policy evidence
    context = build_context(question, top_k=4)

    if not context.strip():
        return (
            "I couldn't find reliable policy information. "
            "I'll need to check with support."
        )

    return (
        "Relevant policy evidence was retrieved, but the answer-generation "
        "step still needs to be connected to an LLM."
    )


In [ ]:
print(support_agent(
    "My product arrived damaged. Please create a support ticket.",
    create_ticket_requested=True
))

In [ ]:
print(support_agent(
    "Where is my order?",
    order_id="69520867"
))

In [ ]:
print(support_agent(
    "Where is my order?",
    order_id="69520867",
    verification_value="PASTE_VERIFICATION_VALUE_HERE"
))

In [ ]:
!pip -q install groq

In [ ]:
import os
from getpass import getpass
from groq import Groq

os.environ["GROQ_API_KEY"] = getpass("Enter Groq API key: ")

client = Groq(api_key=os.environ["GROQ_API_KEY"])

print("Groq client configured successfully!")

In [ ]:
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful customer support assistant."
        },
        {
            "role": "user",
            "content": "Say hello and briefly introduce yourself."
        }
    ],
    temperature=0.2,
    max_tokens=100
)

print(response.choices[0].message.content)

In [ ]:
def generate_answer(question):
    context = build_context(question, top_k=4)

    if not context.strip():
        return "I couldn't find reliable policy evidence to answer that."

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are ZentroMart's customer support assistant. "
                    "Answer only using the supplied policy context. "
                    "Treat the context as the source of truth. "
                    "Do not invent policies, fees, dates, eligibility, or promises. "
                    "Check whether the customer has provided all facts required "
                    "by the policy before confirming eligibility or an outcome. "
                    "If important facts are missing, ask a concise clarifying question. "
                    "If the policy does not support an answer, say you cannot confirm "
                    "it and recommend contacting support. "
                    "Never offer compensation beyond what the policy explicitly allows. "
                    "Be concise, friendly, and clear."
                )
            },
            {
                "role": "user",
                "content": (
                    f"Policy context:\n{context}\n\n"
                    f"Customer question: {question}"
                )
            }
        ],
        temperature=0.1,
        max_tokens=250
    )

    return response.choices[0].message.content

In [ ]:
print(generate_answer(
    "My jeans arrived damaged. Will I have to pay a reverse pickup fee?"
))

In [ ]:
print(generate_answer(
    "My order arrived at midnight. What exact compensation will I get?"
))

In [ ]:
question = "My order arrived at midnight. What exact compensation will I get?"

context = build_context(question, top_k=4)

print(context)

In [ ]:
print(generate_answer(
    "My Express order arrived after its promised delivery date. "
    "I am not a ZentroMart Plus member. What compensation applies?"
))

In [ ]:
conversation_history = []

def chat_with_agent(question):
    global conversation_history

    context = build_context(question, top_k=4)

    messages = [
        {
            "role": "system",
            "content": (
                "You are ZentroMart's customer support assistant. "
                "Answer only using the supplied policy context. "
                "Remember relevant details from earlier conversation turns. "
                "Ask concise clarifying questions when required facts are missing. "
                "Never invent policies, fees, eligibility, timelines, or follow-up promises. "
                "Do not offer actions or guarantees that are not explicitly supported "
                "by the policy context. "
                "If evidence is insufficient, say so and recommend contacting support. "
                "Be concise, friendly, and clear."
            )
        }
    ]

    messages.extend(conversation_history[-6:])

    messages.append({
        "role": "user",
        "content": (
            f"Policy context:\n{context}\n\n"
            f"Customer question: {question}"
        )
    })

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=messages,
        temperature=0.1,
        max_tokens=250
    )

    answer = response.choices[0].message.content

    conversation_history.append({"role": "user", "content": question})
    conversation_history.append({"role": "assistant", "content": answer})

    return answer

In [ ]:
print(chat_with_agent(
    "My order arrived late. What compensation can I get?"
))

In [ ]:
print(chat_with_agent(
    "It was Express, arrived after the promised date, and I'm not a Plus member."
))

In [ ]:
print(chat_with_agent(
    "My order arrived late. What compensation can I get?"
))

In [ ]:
print(chat_with_agent(
    "It was Express, arrived after the promised date, and I'm not a Plus member."
))

In [ ]:
import groq
print(groq.__version__)

In [ ]:
order_lookup_tool = {
    "type": "function",
    "function": {
        "name": "lookup_order",
        "description": (
            "Look up an order by its order ID. "
            "Use this when a customer asks about a specific order. "
            "Do not reveal order details until the customer's identity "
            "has been verified."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "The customer's order ID"
                }
            },
            "required": ["order_id"],
            "additionalProperties": False
        }
    }
}

print("Order lookup tool schema ready!")

In [ ]:
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "system",
            "content": (
                "You are ZentroMart support. "
                "If the customer asks about a specific order and gives "
                "an order ID, request the lookup_order tool. "
                "Do not claim you have looked up an order unless the tool runs."
            )
        },
        {
            "role": "user",
            "content": "Please check order 69520867."
        }
    ],
    tools=[order_lookup_tool],
    tool_choice="auto",
    temperature=0.1
)

print(response.choices[0].message.tool_calls)

In [ ]:
import json

tool_call = response.choices[0].message.tool_calls[0]

if tool_call.function.name == "lookup_order":
    arguments = json.loads(tool_call.function.arguments)

    order_id = arguments["order_id"]

    result = lookup_order(order_id)

    print("Tool requested:", tool_call.function.name)
    print("Tool result:", result)

In [ ]:
tool_message = {
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": json.dumps({
        "found": result["found"],
        "order_status": result["order"]["status"],
        "items": [
            item["title"] for item in result["items"]
        ]
    })
}

messages = [
    {
        "role": "system",
        "content": (
            "You are ZentroMart support. "
            "Summarize tool results clearly. "
            "Never reveal order details unless identity verification "
            "has been completed successfully."
        )
    },
    {
        "role": "user",
        "content": "Please check order 69520867."
    },
    response.choices[0].message,
    tool_message
]

final_response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=messages,
    temperature=0.1,
    max_tokens=150
)

print(final_response.choices[0].message.content)

In [ ]:
# Test the verification gate before revealing order details
verification = verify_customer("69520867", "WRONG_VALUE")

if not verification["verified"]:
    print("Access blocked: identity verification required.")
else:
    print("Access allowed: identity verified.")

In [ ]:
# Session state for secure order verification
pending_order_id = None
verification_attempts = 0
verified_orders = set()

print("Secure verification session initialized.")

In [ ]:
def request_order_verification(order_id, verification_value):
    global pending_order_id, verification_attempts, verified_orders

    # Start a fresh verification flow for a different order
    if pending_order_id != order_id:
        pending_order_id = order_id
        verification_attempts = 0

    # Already verified in this session
    if order_id in verified_orders:
        return {
            "verified": True,
            "message": "Identity already verified."
        }

    # Allow at most two attempts
    if verification_attempts >= 2:
        return {
            "verified": False,
            "message": "Verification attempts exhausted. Please contact support."
        }

    verification_attempts += 1
    result = verify_customer(order_id, verification_value)

    if result["verified"]:
        verified_orders.add(order_id)
        return {
            "verified": True,
            "message": "Identity verified successfully."
        }

    remaining = 2 - verification_attempts

    return {
        "verified": False,
        "message": (
            "Verification failed. One retry remains."
            if remaining == 1
            else "Verification attempts exhausted. Please contact support."
        ),
        "attempts_remaining": remaining
    }

print("Verification-first workflow ready.")

In [ ]:
def secure_order_lookup(order_id, verification_value=None):
    # Do not fetch or reveal order details before verification
    if order_id not in verified_orders:
        if verification_value is None:
            return {
                "success": False,
                "message": (
                    "Identity verification required. "
                    "Provide your registered phone's last 4 digits "
                    "or delivery pincode."
                )
            }

        verification = request_order_verification(
            order_id, verification_value
        )

        if not verification["verified"]:
            return {
                "success": False,
                "message": verification["message"],
                "attempts_remaining": verification.get(
                    "attempts_remaining", 0
                )
            }

    # Only fetch order details after successful verification
    result = lookup_order(order_id)

    if not result["found"]:
        return {
            "success": False,
            "message": "Order not found."
        }

    return {
        "success": True,
        "order": result["order"],
        "items": result["items"]
    }


print("Secure order lookup ready.")

In [ ]:
# Test 1: No verification value supplied
test_result = secure_order_lookup("69520867")

print(test_result)

In [ ]:
test_result = secure_order_lookup("69520867", "WRONG_VALUE")
print(test_result)

In [ ]:
test_result = secure_order_lookup("69520867", "WRONG_VALUE")
print(test_result)

In [ ]:
print("Python is running!")
print("client exists:", "client" in globals())
print("documents exist:", "documents" in globals())
print("lookup_order exists:", "lookup_order" in globals())
print("secure_order_tool exists:", "secure_order_tool" in globals())

In [ ]:
from pathlib import Path

project = Path(
    "/content/voiceops_data/Capstone_Project_Dataset_FTTT_2026"
)

print("Dataset folder exists:", project.exists())

if project.exists():
    print("Data folder exists:", (project / "data").exists())
    print("CSV files:", len(list((project / "data").glob("*.csv"))))

In [ ]:
from pathlib import Path
import zipfile

# Find the uploaded dataset ZIP
zip_files = list(Path("/content").rglob(
    "Capstone_Project_Dataset_FTTT_2026.zip"
))

print("ZIP files found:", len(zip_files))

for f in zip_files:
    print(f)

In [ ]:
from pathlib import Path
import zipfile

zip_path = zip_files[0]
extract_to = Path("/content/voiceops_data")
extract_to.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_to)

project = extract_to / "Capstone_Project_Dataset_FTTT_2026"

print("Dataset extracted:", project.exists())
print("CSV files:", len(list(project.rglob("*.csv"))))
print("Markdown files:", len(list(project.rglob("*.md"))))

In [ ]:
!pip -q install sentence-transformers groq

from pathlib import Path
import pandas as pd
import re
import os
from getpass import getpass
from sentence_transformers import SentenceTransformer
from groq import Groq

# 1. Find the actual dataset folders
csv_matches = list(project.rglob("orders.csv"))
assert csv_matches, "orders.csv not found in extracted dataset!"

csv_folder = csv_matches[0].parent
print("CSV folder:", csv_folder)

# 2. Load all CSV files as strings (preserves IDs and pincodes)
dataframes = {}
for file in csv_folder.glob("*.csv"):
    dataframes[file.stem] = pd.read_csv(file, dtype=str).fillna("")

customers = dataframes["customers"]
orders = dataframes["orders"]
order_items = dataframes["order_items"]
products = dataframes["products"]
pincodes = dataframes["pincodes"]
holidays = dataframes["holidays"]
support_logs = dataframes["support_logs"]

print("CSV files loaded:", len(dataframes))

# 3. Reload policy documents
documents = []
for file in project.rglob("*.md"):
    documents.append({
        "filename": file.name,
        "text": file.read_text(encoding="utf-8")
    })

print("Markdown files loaded:", len(documents))

# 4. Rebuild policy chunks
chunks = []
for doc in documents:
    sections = re.split(r"(?=^##\s+\[)", doc["text"], flags=re.MULTILINE)
    for section in sections:
        section = section.strip()
        if section:
            chunks.append({
                "filename": doc["filename"],
                "text": section
            })

chunk_texts = [
    f"File: {c['filename']}\n{c['text']}" for c in chunks
]
print("Policy chunks:", len(chunk_texts))

# 5. Rebuild embeddings
embedding_model = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2"
)
multilingual_embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True,
    normalize_embeddings=True
)

# 6. Reconnect Groq securely
os.environ["GROQ_API_KEY"] = getpass("YOUR_API_KEY_REMOVED ")
client = Groq(api_key=os.environ["GROQ_API_KEY"])

print("\nRECOVERY COMPLETE")
print("Orders:", len(orders))
print("Policy chunks:", len(chunk_texts))
print("Embeddings shape:", multilingual_embeddings.shape)
print("Groq client: ready")

In [ ]:
print("Orders loaded:", "orders" in globals())
print("Customers loaded:", "customers" in globals())
print("Policy chunks:", "chunk_texts" in globals())
print("Embeddings loaded:", "multilingual_embeddings" in globals())
print("Groq client ready:", "client" in globals())

In [ ]:
import uuid

# Order lookup: IDs remain strings
def lookup_order(order_id):
    order_match = orders[orders["order_id"] == str(order_id)]

    if order_match.empty:
        return {"found": False}

    order = order_match.iloc[0].to_dict()
    items = order_items[
        order_items["order_id"] == str(order_id)
    ].to_dict("records")

    return {
        "found": True,
        "order": order,
        "items": items
    }


# Verify using registered phone's last 4 digits OR delivery pincode
def verify_customer(order_id, verification_value):
    result = lookup_order(order_id)

    if not result["found"]:
        return {"verified": False, "message": "Order not found."}

    order = result["order"]
    match = customers[
        customers["customer_id"] == order["customer_id"]
    ]

    if match.empty:
        return {"verified": False, "message": "Customer record not found."}

    customer = match.iloc[0]
    supplied = str(verification_value).strip()

    if supplied in (customer["phone"][-4:], order["delivery_pincode"]):
        return {"verified": True, "message": "Identity verified."}

    return {"verified": False, "message": "Verification failed."}


# In-memory ticket creation
tickets = []

def create_ticket(summary):
    ticket_id = "ZT-" + uuid.uuid4().hex[:8].upper()
    tickets.append({
        "ticket_id": ticket_id,
        "summary": str(summary),
        "created": True
    })

    return {
        "created": True,
        "ticket_id": ticket_id
    }


print("Order lookup restored:", lookup_order("69520867")["found"])
print("Verification function restored:", callable(verify_customer))
print("Ticket creation restored:", callable(create_ticket))

In [ ]:
# Secure session state
pending_order_id = None
verification_attempts = 0
verified_orders = set()

def secure_order_lookup(order_id, verification_value=None):
    global pending_order_id, verification_attempts, verified_orders

    order_id = str(order_id)

    # Require verification before returning any order details
    if order_id not in verified_orders:
        if pending_order_id != order_id:
            pending_order_id = order_id
            verification_attempts = 0

        if verification_value is None:
            return {
                "success": False,
                "needs_verification": True,
                "message": (
                    "Please verify using your registered phone's "
                    "last 4 digits or delivery pincode."
                )
            }

        if verification_attempts >= 2:
            return {
                "success": False,
                "message": "Verification attempts exhausted. Please contact support."
            }

        verification_attempts += 1
        result = verify_customer(order_id, verification_value)

        if not result["verified"]:
            remaining = 2 - verification_attempts
            return {
                "success": False,
                "message": (
                    "Verification failed. One retry remains."
                    if remaining == 1
                    else "Verification attempts exhausted. Please contact support."
                ),
                "attempts_remaining": remaining
            }

        verified_orders.add(order_id)

    # Only now retrieve the order details
    result = lookup_order(order_id)

    if not result["found"]:
        return {"success": False, "message": "Order not found."}

    return {
        "success": True,
        "order": result["order"],
        "items": result["items"]
    }

print("Secure order workflow restored.")

In [ ]:
import json

tools = [
    {
        "type": "function",
        "function": {
            "name": "lookup_order",
            "description": (
                "Request an order lookup. Python will require identity "
                "verification before returning any order details."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {"type": "string"}
                },
                "required": ["order_id"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "create_ticket",
            "description": (
                "Create a support ticket when the issue needs escalation. "
                "Only confirm creation if the tool returns created=true."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "summary": {"type": "string"}
                },
                "required": ["summary"],
                "additionalProperties": False
            }
        }
    }
]

print("Tools ready:", [t["function"]["name"] for t in tools])

In [ ]:
def execute_tool(tool_name, arguments):
    if tool_name == "lookup_order":
        # Never call raw lookup_order directly from the LLM executor
        return secure_order_lookup(
            order_id=arguments["order_id"]
        )

    elif tool_name == "create_ticket":
        result = create_ticket(arguments["summary"])

        # Only report success if the tool confirms it
        if result.get("created") is True:
            return {
                "created": True,
                "ticket_id": result["ticket_id"]
            }

        return {
            "created": False,
            "message": "Ticket creation failed."
        }

    return {
        "success": False,
        "message": "Unknown tool."
    }


print("Tool executor ready.")

In [ ]:
def run_agent_tool_call(user_message):
    messages = [
        {
            "role": "system",
            "content": (
                "You are ZentroMart customer support. "
                "Use tools when needed. Never claim an order was checked "
                "unless the tool ran. Never reveal order details unless "
                "identity verification has succeeded. "
                "If a tool requests verification, ask the customer for "
                "their registered phone's last 4 digits or delivery pincode. "
                "Never ask for a full phone number, password, or OTP. "
                "Only confirm ticket creation when created=true."
            )
        },
        {"role": "user", "content": user_message}
    ]

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=messages,
        tools=tools,
        tool_choice="auto",
        temperature=0.1,
        max_tokens=250
    )

    message = response.choices[0].message

    if not message.tool_calls:
        return message.content

    messages.append(message)

    for tool_call in message.tool_calls:
        name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        # Execute through our Python-controlled gate
        result = execute_tool(name, arguments)

        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(result)
        })

    final_response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=messages,
        temperature=0.1,
        max_tokens=250
    )

    return final_response.choices[0].message.content


print("LLM tool-calling loop ready.")

In [ ]:
answer = run_agent_tool_call(
    "Please check the status of order 69520867."
)

print(answer)

In [ ]:
# Persistent conversation history for the current runtime
agent_history = []

def run_agent_turn(user_message):
    global agent_history

    agent_history.append({
        "role": "user",
        "content": user_message
    })

    # Keep recent conversation context
    recent_history = agent_history[-8:]

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are ZentroMart customer support. "
                    "Use tools when needed. Never reveal order details "
                    "before Python confirms identity verification. "
                    "Ask only for the registered phone's last 4 digits "
                    "or delivery pincode. Never ask for a full phone number, "
                    "password, or OTP. Only confirm ticket creation if "
                    "the tool returns created=true."
                )
            },
            *recent_history
        ],
        tools=tools,
        tool_choice="auto",
        temperature=0.1,
        max_tokens=250
    )

    message = response.choices[0].message

    if message.tool_calls:
        agent_history.append({
            "role": "assistant",
            "content": message.content or ""
        })
        answer = (
            "I need to process the requested action securely. "
            "Please continue with the verification prompt if requested."
        )
    else:
        answer = message.content or ""

    agent_history.append({
        "role": "assistant",
        "content": answer
    })

    return answer

print("Multi-turn session memory ready.")

In [ ]:
agent_history = []
pending_verification_order = None

def support_turn(user_message):
    global agent_history, pending_verification_order

    user_message = user_message.strip()

    # Handle a verification reply in Python
    if pending_verification_order is not None:
        order_id = pending_verification_order
        result = secure_order_lookup(order_id, user_message)

        if result.get("success"):
            pending_verification_order = None

            safe_order = {
                "status": result["order"].get("status"),
                "promised_date": result["order"].get("promised_date"),
                "items": [item.get("title", "") for item in result["items"]]
            }

            response = client.chat.completions.create(
                model="openai/gpt-oss-120b",
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "You are concise customer support. "
                            "Use only the verified order details provided."
                        )
                    },
                    {
                        "role": "user",
                        "content": (
                            "The customer has been verified. "
                            "Answer their order question using these details: "
                            + json.dumps(safe_order)
                        )
                    }
                ],
                temperature=0.1,
                max_tokens=180
            )
            answer = response.choices[0].message.content or ""

        else:
            answer = result.get("message", "Verification failed.")
            if result.get("attempts_remaining", 1) == 0:
                pending_verification_order = None

        agent_history.extend([
            {"role": "user", "content": "[verification response]"},
            {"role": "assistant", "content": answer}
        ])
        return answer

    # Ask the LLM whether a tool is needed
    messages = [
        {
            "role": "system",
            "content": (
                "You are ZentroMart support. Use lookup_order for specific "
                "order-status requests. Never reveal order details before "
                "Python verifies identity. Never ask for a full phone number, "
                "password, or OTP. Only confirm ticket creation if the tool "
                "returns created=true."
            )
        },
        *agent_history[-6:],
        {"role": "user", "content": user_message}
    ]

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=messages,
        tools=tools,
        tool_choice="auto",
        temperature=0.1,
        max_tokens=200
    )

    message = response.choices[0].message

    if message.tool_calls:
        answer = "I couldn't process that request."

        for tool_call in message.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)

            if name == "lookup_order":
                pending_verification_order = str(args["order_id"])
                answer = (
                    "To check that order, provide either the last 4 digits "
                    "of your registered phone number or the delivery pincode."
                )
                break

            if name == "create_ticket":
                result = create_ticket(args["summary"])
                if result.get("created") is True:
                    answer = (
                        "Your support ticket was created. Reference: "
                        + result["ticket_id"]
                    )
                else:
                    answer = (
                        "I couldn't confirm ticket creation. Please try again."
                    )
                break
    else:
        answer = message.content or "Sorry, I couldn't generate a response."

    agent_history.extend([
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": answer}
    ])

    return answer


print("Secure multi-turn support handler ready.")

In [ ]:
print(support_turn("Please check order 69520867."))

In [ ]:
print(support_turn("WRONG_VALUE"))

In [ ]:
print(support_turn("WRONG_VALUE"))

In [ ]:

def support_turn_v2(user_message):
    global support_history, support_session

    text = user_message.strip()
    lower = text.lower()

    # 1. Explicit ticket/escalation requests take priority
    escalation_terms = (
        "create a ticket", "raise a ticket", "escalate",
        "human agent", "support agent", "file a complaint"
    )

    if any(term in lower for term in escalation_terms):
        result = create_support_ticket(text)

        if result.get("created") is True:
            answer = (
                "Your support ticket was created. Reference: "
                + result["ticket_id"] + "."
            )
        else:
            answer = (
                "I couldn't confirm ticket creation. "
                "Please contact support."
            )

        support_history.extend([
            {"role": "user", "content": text},
            {"role": "assistant", "content": answer}
        ])
        return answer

    # 2. Handle a pending order verification
    pending_id = support_session["pending_order_id"]

    if pending_id is not None:
        if support_session["verification_attempts"] >= 2:
            support_session["pending_order_id"] = None
            return "Verification attempts exhausted. Please contact support."

        support_session["verification_attempts"] += 1
        check = verify_customer(pending_id, text)

        if not check.get("verified"):
            remaining = 2 - support_session["verification_attempts"]

            if remaining == 0:
                support_session["pending_order_id"] = None
                return "Verification attempts exhausted. Please contact support."

            return "Verification failed. One retry remains."

        # Only fetch order details after verification succeeds
        order_result = lookup_order(pending_id)
        support_session["pending_order_id"] = None
        support_session["verification_attempts"] = 0

        if not order_result.get("found"):
            return "I couldn't find that order. Please check the order ID."

        order = order_result["order"]
        item_titles = [
            item.get("title", "") for item in order_result["items"]
        ]

        answer = (
            "Your identity is verified. "
            f"Order status: {order.get('status', 'Unknown')}. "
            f"Promised date: {order.get('promised_date', 'Unknown')}. "
            f"Items: {', '.join(item_titles) or 'Not listed'}."
        )

        support_history.extend([
            {"role": "user", "content": "[verification response]"},
            {"role": "assistant", "content": answer}
        ])
        return answer

    # 3. Order requests must pass through verification
    order_intent = re.search(
        r"\b(order|delivery|shipment|track|tracking|where is)\b",
        lower
    )
    order_id_match = re.search(r"\b(\d{8})\b", text)

    if order_intent and order_id_match:
        support_session["pending_order_id"] = order_id_match.group(1)
        support_session["verification_attempts"] = 0

        return (
            "To check that order, provide either the last 4 digits "
            "of your registered phone number or the delivery pincode."
        )

    # 4. Policy questions use RAG
    context = retrieve_policy_context(text, top_k=4)

    messages = [{
        "role": "system",
        "content": (
            "You are ZentroMart customer support. Answer using only the "
            "supplied policy context. Do not invent policies, fees, eligibility, "
            "or timelines. If evidence is insufficient, say so and recommend "
            "contacting support. Never claim a ticket was created unless "
            "Python confirms it. Never reveal order details from conversation history."
        )
    }]

    messages.extend(support_history[-6:])
    messages.append({
        "role": "user",
        "content": f"Policy context:\n{context}\n\nCustomer question: {text}"
    })

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=messages,
        temperature=0.1,
        max_tokens=250
    )

    answer = response.choices[0].message.content or (
        "Sorry, I couldn't generate a response."
    )

    support_history.extend([
        {"role": "user", "content": text},
        {"role": "assistant", "content": answer}
    ])

    return answer


print("Support handler updated.")


In [ ]:
support_session = {
    "pending_order_id": "69520867",
    "verification_attempts": 1
}

print(support_turn_v2("Please create a support ticket for my issue."))

In [ ]:
import inspect

print(inspect.getsource(support_turn_v2)[:1200])

In [ ]:
# Reset state between independent demo scenarios
def reset_support_session():
    global support_history, support_session

    support_history = []
    support_session = {
        "pending_order_id": None,
        "verification_attempts": 0
    }

reset_support_session()
print("Session reset.")

In [ ]:
# Reset ticket test state
support_history = []
support_session = {
    "pending_order_id": "69520867",
    "verification_attempts": 1
}

result = support_turn_v2("Please create a support ticket for my issue.")
print(result)

In [ ]:
print("Handler currently loaded:")
print(support_turn_v2.__code__.co_firstlineno)

print("\nTicket phrase match:")
text = "Please create a support ticket for my issue."
lower = text.lower()
escalation_terms = (
    "create a ticket", "raise a ticket", "escalate",
    "human agent", "support agent", "file a complaint"
)
print(any(term in lower for term in escalation_terms))

print("\nTicket function:")
print(create_support_ticket("Diagnostic test"))

In [ ]:
escalation_terms = (
    "create a ticket",
    "create a support ticket",
    "raise a ticket",
    "raise a support ticket",
    "escalate",
    "human agent",
    "support agent",
    "file a complaint",
    "speak to a human"
)

print(any(
    term in "please create a support ticket for my issue."
    for term in escalation_terms
))

In [ ]:
def handle_ticket_request(summary):
    result = create_ticket(summary)

    if result.get("created") is True and result.get("ticket_id"):
        return f"Ticket created successfully. Reference: {result['ticket_id']}"

    return "Ticket creation could not be confirmed. Please contact support."


print(handle_ticket_request("Customer requests help with an unresolved issue."))

In [ ]:
Verification attempts exhausted. Please contact support.

In [ ]:
# Test 1: Policy question
support_session = {
    "pending_order_id": None,
    "verification_attempts": 0
}
support_history = []

print("POLICY:")
print(support_turn_v2("What is the return policy?"))

print("\nORDER VERIFICATION:")
print(support_turn_v2("Please check order 69520867."))

In [ ]:
# DAY 3 — PROJECT RECOVERY CHECK
required = [
    "orders",
    "customers",
    "order_items",
    "chunk_texts",
    "multilingual_embeddings",
    "client",
    "support_turn_v2"
]

for name in required:
    print(f"{name}: {'READY' if name in globals() else 'MISSING'}")

In [ ]:
# DAY 3 — RESTORE PROJECT AFTER COLAB RESET
!pip -q install sentence-transformers groq

import os, glob, zipfile, getpass
import pandas as pd
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer
from groq import Groq

# 1. Find dataset ZIP; upload it if it is no longer in Colab
zips = glob.glob("/content/**/*.zip", recursive=True)
zips = [z for z in zips if "Capstone_Project_Dataset_FTTT_2026" in os.path.basename(z)]

if not zips:
    from google.colab import files
    print("Please upload Capstone_Project_Dataset_FTTT_2026.zip")
    uploaded = files.upload()
    zips = ["/content/" + name for name in uploaded
            if name.lower().endswith(".zip")]

if not zips:
    raise FileNotFoundError("Dataset ZIP not found. Upload the capstone ZIP.")

zip_path = zips[0]
extract_root = "/content/voiceops_data"
os.makedirs(extract_root, exist_ok=True)

with zipfile.ZipFile(zip_path) as z:
    z.extractall(extract_root)

roots = list(Path(extract_root).rglob("Capstone_Project_Dataset_FTTT_2026"))
if not roots:
    roots = [Path(extract_root)]

project = roots[0]
csv_folder = project / "data"
if not csv_folder.exists():
    raise FileNotFoundError("Could not locate the dataset's data folder.")

# 2. Load CSVs with IDs preserved as strings
tables = {}
for f in csv_folder.glob("*.csv"):
    tables[f.stem] = pd.read_csv(f, dtype=str).fillna("")

orders = tables["orders"]
customers = tables["customers"]
order_items = tables["order_items"]
products = tables["products"]
pincodes = tables["pincodes"]
holidays = tables["holidays"]
support_logs = tables["support_logs"]

print("CSV tables loaded:", list(tables.keys()))

# 3. Load knowledge-base Markdown files, excluding superseded policy
kb_folder = project / "kb"
if not kb_folder.exists():
    kb_folder = project

documents = []
for f in kb_folder.rglob("*.md"):
    if "superseded" in f.name.lower():
        continue
    documents.append({
        "source": f.name,
        "text": f.read_text(encoding="utf-8", errors="ignore")
    })

chunk_texts = []
chunk_sources = []

for doc in documents:
    text = doc["text"]
    # Simple overlapping chunks for recovery; preserve source filename
    step = 850
    size = 1000
    for start in range(0, len(text), step):
        chunk = text[start:start + size].strip()
        if chunk:
            chunk_texts.append(chunk)
            chunk_sources.append(doc["source"])

if not chunk_texts:
    raise ValueError("No Markdown knowledge documents found.")

# 4. Rebuild embeddings
embedding_model = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2"
)
multilingual_embeddings = embedding_model.encode(
    chunk_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

# 5. Reconnect Groq privately
groq_key = getpass.getpass("Enter Groq API key: ")
client = Groq(api_key=groq_key)
del groq_key

# 6. Restore basic lookup and ticket helpers
def lookup_order(order_id):
    match = orders[orders["order_id"] == str(order_id)]
    if match.empty:
        return {"found": False}
    order = match.iloc[0].to_dict()
    items = order_items[
        order_items["order_id"] == str(order_id)
    ].to_dict("records")
    return {"found": True, "order": order, "items": items}

tickets = []

def create_ticket(summary):
    import uuid
    ticket_id = "ZT-" + uuid.uuid4().hex[:8].upper()
    tickets.append({"ticket_id": ticket_id, "summary": summary})
    return {"created": True, "ticket_id": ticket_id}

print("\nRECOVERY COMPLETE")
print("Orders:", len(orders))
print("Customers:", len(customers))
print("Policy chunks:", len(chunk_texts))
print("Embeddings:", multilingual_embeddings.shape)
print("Groq client: connected")
print("NOTE: support_turn_v2 and conversation state still need restoration.")

In [ ]:
checks = [
    "lookup_order",
    "verify_customer",
    "create_ticket",
    "create_support_ticket",
    "retrieve_policy_context",
    "support_turn_v2",
    "multilingual_embeddings",
    "chunk_texts",
    "client",
]

for name in checks:
    print(f"{name}: {'OK' if name in globals() else 'MISSING'}")

In [ ]:

import inspect
import pandas as pd
import numpy as np

print("=== DATA SCHEMAS ===")
for name in ["orders", "customers", "order_items", "products",
             "pincodes", "holidays", "support_logs"]:
    obj = globals().get(name)
    if isinstance(obj, pd.DataFrame):
        print(f"\n{name}: {obj.shape}")
        print("Columns:", obj.columns.tolist())

print("\n=== FUNCTION SIGNATURES ===")
for name in ["lookup_order", "verify_customer", "create_ticket",
             "create_support_ticket", "retrieve_policy_context",
             "support_turn_v2"]:
    fn = globals().get(name)
    if callable(fn):
        try:
            print(name, inspect.signature(fn))
            print(inspect.getsource(fn)[:1800])
        except Exception as e:
            print(name, "source unavailable:", str(e))
    else:
        print(name, "MISSING")

print("\n=== EMBEDDING OBJECTS ===")
for name in ["embedding_model", "model", "multilingual_embeddings",
             "chunk_texts", "chunk_sources", "documents"]:
    obj = globals().get(name)
    if obj is None:
        print(name, "MISSING")
    elif hasattr(obj, "shape"):
        print(name, "shape =", obj.shape)
    else:
        try:
            print(name, "type =", type(obj).__name__, "length =", len(obj))
        except Exception:
            print(name, "type =", type(obj).__name__)

print("\n=== TICKET TEST (NO TICKET CREATED) ===")
print("create_ticket callable:", callable(globals().get("create_ticket")))
print("Current ticket function source:")
if callable(globals().get("create_ticket")):
    try:
        print(inspect.getsource(create_ticket))
    except Exception as e:
        print("Source unavailable:", e)


In [ ]:

# ============================================================
# VOICEOPS 2026 — CONSOLIDATED CORE AGENT
# Reuses existing dataframes, embeddings and Groq client.
# ============================================================

import re
import uuid
import numpy as np
from datetime import datetime

# ---------- 1. Conversation state ----------
conversation_state = {}

# ---------- 2. Customer identity verification ----------
def verify_customer(order_id, answer):
    order_id, answer = str(order_id).strip(), str(answer).strip()
    result = lookup_order(order_id)

    if not result.get("found"):
        return False

    order = result["order"]
    # Accept either this order's delivery pincode or registered
    # customer's phone last four digits.
    if answer == str(order.get("delivery_pincode", "")).strip():
        return True

    customer_id = str(order.get("customer_id", "")).strip()
    rows = customers[customers["customer_id"].astype(str) == customer_id]

    if not rows.empty:
        phone = re.sub(r"\D", "", str(rows.iloc[0].get("phone", "")))
        supplied = re.sub(r"\D", "", answer)
        if len(phone) >= 4 and len(supplied) == 4:
            return supplied == phone[-4:]

    return False


# ---------- 3. Ticket creation ----------
def create_support_ticket(summary):
    summary = str(summary).strip()
    if len(summary) < 8:
        return {"created": False, "error": "Please provide more detail."}
    result = create_ticket(summary)
    return result if isinstance(result, dict) else {
        "created": False, "error": "Ticket creation was not confirmed."
    }


# ---------- 4. RAG policy retrieval ----------
def retrieve_policy_context(query, top_k=4):
    if not chunk_texts or len(multilingual_embeddings) == 0:
        return []

    qvec = embedding_model.encode(
        [str(query)], normalize_embeddings=True
    )[0]
    matrix = np.asarray(multilingual_embeddings)
    scores = matrix @ qvec
    # Avoid irrelevant low-similarity chunks.
    indices = np.argsort(scores)[::-1][:top_k]

    return [
        {
            "text": chunk_texts[int(i)],
            "source": chunk_sources[int(i)],
            "score": float(scores[int(i)]),
        }
        for i in indices
        if float(scores[int(i)]) >= 0.20
    ]


# ---------- 5. Order lookup tool ----------
def lookup_order_safe(order_id):
    result = lookup_order(str(order_id))
    if not result.get("found"):
        return {"found": False}
    return result


# ---------- 6. Main multi-turn handler ----------
def support_turn_v2(session_id, user_message):
    sid = str(session_id)
    msg = str(user_message).strip()
    low = msg.lower()

    state = conversation_state.setdefault(sid, {
        "history": [],
        "pending_verification": None,
        "verification_attempts": 0,
        "verified_orders": set(),
        "pending_ticket": False,
    })

    if not msg:
        return {"reply": "Please tell me how I can help.", "retrieved_chunks": []}

    # If we are waiting for verification, treat this turn as the answer.
    pending = state["pending_verification"]
    if pending:
        order_id = pending
        if verify_customer(order_id, msg):
            state["verified_orders"].add(order_id)
            state["pending_verification"] = None
            state["verification_attempts"] = 0
            result = lookup_order_safe(order_id)
            # Only reveal limited order details after successful verification.
            order = result["order"]
            reply = (
                f"Verification successful. Order {order_id} status: "
                f"{order.get('status', 'unavailable')}. "
                f"Promised delivery date: {order.get('promised_date', 'unavailable')}."
            )
        else:
            state["verification_attempts"] += 1
            state["pending_verification"] = None
            if state["verification_attempts"] >= 1:
                reply = (
                    "I couldn't verify those details. For your security, "
                    "please contact customer support for further assistance."
                )
                state["verification_attempts"] = 0
            else:
                state["pending_verification"] = order_id
                reply = "That didn't match. Please try once more using the delivery pincode or phone's last four digits."
        state["history"].append({"role": "user", "content": msg})
        state["history"].append({"role": "assistant", "content": reply})
        return {"reply": reply, "retrieved_chunks": []}

    # Explicit ticket intent. Ask for a description if none is provided.
    ticket_intent = any(x in low for x in [
        "create a ticket", "raise a ticket", "support ticket",
        "create ticket", "escalate this", "raise this issue"
    ])
    if state["pending_ticket"]:
        ticket_result = create_support_ticket(msg)
        state["pending_ticket"] = False
        if ticket_result.get("created") is True:
            reply = f"Your support ticket has been created. Reference: {ticket_result.get('ticket_id')}."
        else:
            reply = "I couldn't confirm ticket creation. Please try again or contact support."
        return {"reply": reply, "retrieved_chunks": []}

    if ticket_intent:
        summary = re.sub(
            r"\b(create|raise|open)\s+(a\s+)?(support\s+)?ticket\b",
            "", msg, flags=re.I
        ).strip(" .,:-")
        if len(summary) < 8:
            state["pending_ticket"] = True
            return {
                "reply": "Please briefly describe the issue you want included in the support ticket.",
                "retrieved_chunks": []
            }
        ticket_result = create_support_ticket(summary)
        if ticket_result.get("created") is True:
            reply = f"Your support ticket has been created. Reference: {ticket_result.get('ticket_id')}."
        else:
            reply = "I couldn't confirm ticket creation. Please contact support."
        return {"reply": reply, "retrieved_chunks": []}

    # Find an order ID mentioned in the message.
    order_id = None
    for oid in orders["order_id"].astype(str):
        if oid and re.search(r"(?<![A-Za-z0-9])" + re.escape(oid) +
                             r"(?![A-Za-z0-9])", msg, re.I):
            order_id = oid
            break

    if order_id:
        if not lookup_order(order_id).get("found"):
            reply = "I couldn't find that order ID. Please check it and try again."
        elif order_id not in state["verified_orders"]:
            state["pending_verification"] = order_id
            state["verification_attempts"] = 0
            reply = (
                f"Before I share details for order {order_id}, please verify "
                "using the delivery pincode or the last four digits of your registered phone number."
            )
        else:
            order = lookup_order(order_id)["order"]
            reply = (
                f"Order {order_id} status: {order.get('status', 'unavailable')}. "
                f"Promised delivery date: {order.get('promised_date', 'unavailable')}."
            )
        state["history"].append({"role": "user", "content": msg})
        state["history"].append({"role": "assistant", "content": reply})
        return {"reply": reply, "retrieved_chunks": []}

    # Retrieve policy evidence.
    retrieved = retrieve_policy_context(msg)
    context = "\n\n".join(
        f"[Source: {r['source']} | similarity: {r['score']:.3f}]\n{r['text']}"
        for r in retrieved
    )

    system_prompt = (
        "You are VoiceOps, a helpful and concise Indian e-commerce support agent. "
        "Use only the retrieved policy excerpts for policy-specific claims. "
        "If the evidence is missing or ambiguous, say you cannot confirm and offer "
        "to escalate. Never invent refunds, dates, eligibility, order statuses or "
        "ticket IDs. Never claim a ticket was created unless the tool result confirms "
        "created=true. Order details require successful verification. "
        "Keep replies short and natural for spoken delivery.\n\n"
        "RETRIEVED POLICY EXCERPTS:\n"
        + (context if context else "[No sufficiently relevant policy chunks retrieved.]")
    )

    history = state["history"][-8:]
    messages = [{"role": "system", "content": system_prompt}] + history + [
        {"role": "user", "content": msg}
    ]

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=messages,
        temperature=0.2,
    )
    reply = response.choices[0].message.content.strip()

    state["history"].append({"role": "user", "content": msg})
    state["history"].append({"role": "assistant", "content": reply})

    return {
        "reply": reply,
        "retrieved_chunks": [
            {"source": r["source"], "score": round(r["score"], 3), "text": r["text"]}
            for r in retrieved
        ],
    }


print("VoiceOps core agent loaded.")
print("Functions:", callable(verify_customer), callable(create_support_ticket),
      callable(retrieve_policy_context), callable(support_turn_v2))


In [ ]:

# 1. Retrieval sanity check
hits = retrieve_policy_context("return and refund policy")
print("RAG chunks:", len(hits))
for h in hits[:2]:
    print(h["source"], round(h["score"], 3), h["text"][:180].replace("\n", " "))

# 2. Policy question
print("\nPOLICY TEST:")
print(support_turn_v2("test-policy", "What is the return policy?")["reply"])

# 3. Verification prompt only; use a real test order ID from your synthetic dataset
test_order_id = str(orders.iloc[0]["order_id"])
print("\nORDER TEST:")
print("Test order ID:", test_order_id)
print(support_turn_v2("test-order", f"Can you check order {test_order_id}?")["reply"])

# 4. Ticket creation test
print("\nTICKET TEST:")
print(support_turn_v2(
    "test-ticket",
    "Create a support ticket because my package arrived damaged and I need help."
)["reply"])


In [ ]:

# VOICEOPS — targeted fixes; no dataset/model reload

def retrieve_policy_context(query, top_k=5):
    qvec = embedding_model.encode(
        [str(query)], normalize_embeddings=True
    )[0]
    matrix = np.asarray(multilingual_embeddings)
    scores = matrix @ qvec
    q = str(query).lower()

    candidates = []

    for i, score in enumerate(scores):
        source = str(chunk_sources[i])
        source_lower = source.lower()

        # Never retrieve superseded policy material.
        if "superseded" in source_lower or "v3" in source_lower:
            continue

        bonus = 0.0

        if any(word in q for word in ["return", "exchange", "refund"]):
            if "return" in source_lower or "refund" in source_lower:
                bonus += 0.12

        if any(word in q for word in ["delivery", "late", "shipping"]):
            if "delivery" in source_lower or "shipping" in source_lower:
                bonus += 0.10

        if any(word in q for word in ["payment", "cod", "cash on delivery"]):
            if "payment" in source_lower or "cod" in source_lower:
                bonus += 0.10

        # Prefer reverse-pickup help for pickup-related questions.
        if any(word in q for word in [
            "pickup", "reverse pickup", "return collection",
            "पिकअप", "पिक अप", "उठाने"
        ]):
            if "help_reverse_pickup" in source_lower:
                bonus += 0.20

        candidates.append((float(score) + bonus, i, float(score)))

    candidates.sort(reverse=True)

    return [
        {
            "text": chunk_texts[i],
            "source": chunk_sources[i],
            "score": score,
        }
        for _, i, score in candidates[:top_k]
        if score >= 0.25
    ]


def support_turn_v2(session_id, user_message):
    sid = str(session_id)
    msg = str(user_message).strip()
    low = msg.lower()

    state = conversation_state.setdefault(sid, {
        "history": [],
        "pending_verification": None,
        "verification_attempts": 0,
        "verified_orders": set(),
        "pending_ticket": False,
    })

    if not msg:
        return {
            "reply": "Please tell me how I can help.",
            "retrieved_chunks": []
        }

    # 1. Handle pending order verification
    pending = state.get("pending_verification")

    if pending:
        if verify_customer(pending, msg):
            order_id = pending
            state["verified_orders"].add(order_id)
            state["pending_verification"] = None
            state["verification_attempts"] = 0

            order = lookup_order(order_id)["order"]
            reply = (
                f"Verification successful. Order {order_id} status: "
                f"{order.get('status', 'unavailable')}. Promised delivery date: "
                f"{order.get('promised_date', 'unavailable')}."
            )
        else:
            state["verification_attempts"] = (
                state.get("verification_attempts", 0) + 1
            )

            if state["verification_attempts"] <= 1:
                reply = (
                    "Those details didn't match. Please try once more using "
                    "the delivery pincode or the last four digits of your registered phone."
                )
            else:
                state["pending_verification"] = None
                state["verification_attempts"] = 0
                reply = (
                    "I couldn't verify your details after two attempts. "
                    "For your security, please contact customer support."
                )

        state["history"].extend([
            {"role": "user", "content": msg},
            {"role": "assistant", "content": reply},
        ])

        return {"reply": reply, "retrieved_chunks": []}

    # 2. Handle pending ticket description
    if state.get("pending_ticket"):
        result = create_support_ticket(msg)
        state["pending_ticket"] = False

        if result.get("created") is True:
            reply = (
                "Your support ticket has been created. "
                f"Reference: {result.get('ticket_id')}."
            )
        else:
            reply = (
                "I couldn't confirm ticket creation. Please contact support."
            )

        state["history"].extend([
            {"role": "user", "content": msg},
            {"role": "assistant", "content": reply},
        ])

        return {"reply": reply, "retrieved_chunks": []}

    # 3. Detect ticket requests
    ticket_intent = any(x in low for x in [
        "create a ticket", "raise a ticket", "support ticket",
        "create ticket", "escalate this", "raise this issue",
        "टिकट बनाओ", "टिकट बनाइए", "शिकायत दर्ज",
        "शिकायत करना", "एस्केलेट"
    ])

    if ticket_intent:
        summary = re.sub(
            r"\b(create|raise|open)\s+(a\s+)?(support\s+)?ticket\b",
            "", msg, flags=re.I
        ).strip(" .,:-")

        if len(summary) < 8:
            state["pending_ticket"] = True
            reply = "Please briefly describe the issue for the support ticket."

            state["history"].extend([
                {"role": "user", "content": msg},
                {"role": "assistant", "content": reply},
            ])

            return {"reply": reply, "retrieved_chunks": []}

        result = create_support_ticket(summary)

        if result.get("created") is True:
            reply = (
                "Your support ticket has been created. "
                f"Reference: {result.get('ticket_id')}."
            )
        else:
            reply = (
                "I couldn't confirm ticket creation. Please contact support."
            )

        state["history"].extend([
            {"role": "user", "content": msg},
            {"role": "assistant", "content": reply},
        ])

        return {"reply": reply, "retrieved_chunks": []}

    # 4. Find order IDs and require verification
    order_id = None

    for oid in orders["order_id"].astype(str):
        if oid and re.search(
            r"(?<![A-Za-z0-9])" + re.escape(oid) + r"(?![A-Za-z0-9])",
            msg, re.I
        ):
            order_id = oid
            break

    if order_id:
        result = lookup_order(order_id)

        if not result.get("found"):
            reply = (
                "I couldn't find that order ID. Please check it and try again."
            )
        elif order_id not in state["verified_orders"]:
            state["pending_verification"] = order_id
            state["verification_attempts"] = 0
            reply = (
                f"Before I share details for order {order_id}, please verify "
                "using the delivery pincode or the last four digits "
                "of your registered phone."
            )
        else:
            order = result["order"]
            reply = (
                f"Order {order_id} status: "
                f"{order.get('status', 'unavailable')}. "
                f"Promised delivery date: "
                f"{order.get('promised_date', 'unavailable')}."
            )

        state["history"].extend([
            {"role": "user", "content": msg},
            {"role": "assistant", "content": reply},
        ])

        return {"reply": reply, "retrieved_chunks": []}

    # 5. Expand Hindi pickup queries into English policy terminology
    retrieval_query = msg

    hindi_pickup_terms = [
        "पिकअप", "पिक अप", "उठाने",
        "लेने आएगा", "लेने आएंगे", "पार्सल लेने"
    ]

    if any(term in low for term in hindi_pickup_terms):
        retrieval_query += (
            " reverse pickup booking choose a pickup slot "
            "missed pickup rescheduled next business day "
            "return collection"
        )

    retrieved = retrieve_policy_context(retrieval_query)

    context = "\n\n".join(
        f"[Source: {r['source']}]\n{r['text']}"
        for r in retrieved
    )

    # 6. Generate grounded answers in the customer's language
    history = state["history"][-8:]

    system = (
        "You are VoiceOps, a concise e-commerce support agent. "
        "Answer policy questions only from the supplied excerpts. "
        "Do not invent rules, return windows, dates, refunds, order details, "
        "or ticket IDs. If evidence is insufficient, say you cannot confirm "
        "and offer escalation. Keep replies short and natural for spoken delivery. "
        "Reply in the same language as the customer's latest message. "
        "For Hindi messages, answer naturally in Hindi.\n\n"

        "IMPORTANT REFUND RULES:\n"
        "- Distinguish refund initiation from the time money reaches the customer.\n"
        "- For returned items, regular members' refunds are initiated after "
        "quality check, within 2 business days of pickup; Plus members' refunds "
        "are initiated at pickup.\n"
        "- Use the payment-method timeline only after refund initiation.\n"
        "- For prepaid orders refused at the door or not delivered after "
        "3 attempts, refund initiation occurs once the parcel reaches the "
        "warehouse, usually 7 to 10 calendar days after the last delivery attempt.\n"
        "- Do not use a generic payment-method refund timeline as the answer "
        "to when an undelivered order's refund will be initiated.\n"
        "- Bank-transfer timelines for COD refunds must not be generalized "
        "to prepaid orders.\n"
        "- Never claim a ticket was created unless the tool confirms it.\n\n"

        "IMPORTANT PICKUP RULES:\n"
        "- The customer raises a return through My Orders and chooses a pickup slot.\n"
        "- The policy does not specify a fixed waiting time for the initial pickup.\n"
        "- If the executive does not arrive, pickup is rescheduled automatically "
        "to the next business day.\n"
        "- After two missed pickups, the customer can ask support for priority pickup.\n\n"

        "POLICY EXCERPTS:\n"
        + (
            context
            if context
            else "[No sufficiently relevant policy evidence.]"
        )
    )

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "system", "content": system},
            *history,
            {"role": "user", "content": msg},
        ],
        temperature=0.1,
    )

    reply = response.choices[0].message.content.strip()

    state["history"].extend([
        {"role": "user", "content": msg},
        {"role": "assistant", "content": reply},
    ])

    return {
        "reply": reply,
        "retrieved_chunks": [
            {
                "source": r["source"],
                "score": round(r["score"], 3),
                "text": r["text"]
            }
            for r in retrieved
        ],
    }


print("Corrected handler and retrieval loaded.")
print("Pickup-policy sources:", [
    h["source"]
    for h in retrieve_policy_context(
        "reverse pickup booking choose a pickup slot missed pickup"
    )
])


In [ ]:
test_questions = [
    "I returned my item yesterday. When will I get my refund?",
    "I am a ZentroMart Plus member. When will my refund be initiated?",
    "My prepaid order was not delivered. When will I get my money back?",
    "How long does a bank transfer refund take for a COD order?",
    "How long does a bank transfer refund take for a prepaid order?"
]

for i, question in enumerate(test_questions, 1):
    result = support_turn_v2(
        session_id=f"refund_test_{i}",
        user_message=question
    )
    print(f"\nTEST {i}: {question}")
    print("ANSWER:", result["reply"])
    print("SOURCES:", [
        chunk["source"]
        for chunk in result["retrieved_chunks"]
    ])
    print("-" * 70)

In [ ]:

import os
import getpass

os.environ["GNANI_API_KEY"] = getpass.getpass(
    "Paste your Gnani API key (hidden): "
)

print("Gnani key loaded for this Colab session.")


In [ ]:

%pip -q install gnani-vachana

from gnani.stt import GnaniSTTClient
from gnani.tts import GnaniTTSClient

stt = GnaniSTTClient()
tts = GnaniTTSClient()

print("Gnani STT and TTS clients initialised.")


In [ ]:

from pathlib import Path

test_audio = "/content/gnani_test.wav"

tts.synthesize(
    "Hello! Welcome to ZentroMart customer support. How can I help you today?",
    voice="Kaveri",
    model="timbre-v2.5",
    language="en-IN",
    output_file=test_audio
)

assert Path(test_audio).exists(), "Gnani did not create the expected audio file."
print("TTS test passed:", test_audio)


In [ ]:
from IPython.display import Audio, display

display(Audio("/content/gnani_test.wav"))

In [ ]:
import os
import soundfile as sf

test_input = "/content/gnani_test.wav"

print("Audio exists:", os.path.exists(test_input))
print("Audio duration (seconds):", round(len(sf.read(test_input)[0]) / sf.info(test_input).samplerate, 2))

In [ ]:
result = stt.transcribe(
    "/content/gnani_test.wav",
    language_code="en-IN"
)

print("Transcription:", result["transcript"])

In [ ]:
print("Agent function available:", callable(support_turn_v2))

In [ ]:
reply = support_turn_v2(
    session_id="voice_test_001",
    user_message="Hello, I need help with my order."
)

print(reply)

In [ ]:

from pathlib import Path
from gnani.stt import GnaniSTTClient
from gnani.tts import GnaniTTSClient
from IPython.display import Audio, display
import uuid

stt = GnaniSTTClient()
tts = GnaniTTSClient()

def voice_support(audio_path, session_id=None, language="hi-IN"):
    session_id = session_id or str(uuid.uuid4())

    # 1. Speech to text
    result = stt.transcribe(
        str(audio_path),
        language_code=language
    )
    customer_text = result["transcript"].strip()
    print("Customer said:", customer_text)

    # 2. Send text to the support agent
    agent_result = support_turn_v2(
        session_id=session_id,
        user_message=customer_text
    )
    reply = agent_result["reply"]
    print("Agent reply:", reply)

    # 3. Convert the response to speech
    output_path = f"/content/reply_{session_id}.wav"
    tts.synthesize(
        reply,
        voice="Kaveri",
        model="timbre-v2.5",
        language=language,
        output_file=output_path
    )

    print(
        "Retrieved policy chunks:",
        len(agent_result.get("retrieved_chunks", []))
    )
    print("Reply audio:", output_path)
    display(Audio(output_path))

    return {
        "customer_text": customer_text,
        "reply": reply,
        "audio_path": output_path,
        "retrieved_chunks": agent_result.get("retrieved_chunks", [])
    }

print("Voice pipeline ready.")


In [ ]:
result = voice_support(
    "/content/gnani_test.wav",
    session_id="voice_test_001"
)

In [ ]:
result = voice_support(
    "/content/gnani_test.wav",
    session_id="voice_test_002"
)

In [ ]:
from IPython.display import Audio, display
import numpy as np
import soundfile as sf

# A short silent audio file is only a placeholder.
# We'll create a real spoken test using Gnani TTS next.
print("Ready to prepare a customer-query test.")

In [ ]:
customer_query_audio = "/content/customer_query.wav"

tts.synthesize(
    "Hi, I want to know how I can cancel my order.",
    voice="Kaveri",
    model="timbre-v2.5",
    language="en-IN",
    output_file=customer_query_audio
)

print("Customer query audio created:", customer_query_audio)

In [ ]:
result = voice_support(
    "/content/customer_query.wav",
    session_id="cancel_test_001"
)

print("\nFinal customer text:", result["customer_text"])
print("Agent response:", result["reply"])

In [ ]:
from pathlib import Path

p = Path("/content/reply_cancel_test_001.wav")

print("Reply audio exists:", p.exists())
if p.exists():
    print("File size:", round(p.stat().st_size / 1024, 1), "KB")

In [ ]:
test_queries = [
    "What is the return window for an electronics product?",
    "How can I get a refund for a prepaid order?",
    "Can I return an item after the return window has expired?",
    "How long does a bank transfer refund take?"
]

for q in test_queries:
    print("\n" + "=" * 70)
    print("QUESTION:", q)
    result = support_turn_v2(
        session_id="policy_test_" + str(test_queries.index(q)),
        user_message=q
    )
    print("ANSWER:", result["reply"])
    print("SOURCES:", [
        c["source"] for c in result.get("retrieved_chunks", [])
    ])

In [ ]:
from pathlib import Path

for p in Path("/content/voiceops_data/Capstone_Project_Dataset_FTTT_2026/kb").glob("pol_refunds.md"):
    print(p.read_text(encoding="utf-8"))


In [ ]:
import inspect

print(inspect.getsource(support_turn_v2))

In [ ]:
"- For prepaid orders refused at the door or not delivered after "
"3 attempts, refund initiation occurs once the parcel reaches the "
"warehouse, usually 7 to 10 calendar days after the last delivery attempt.\n"
"- Do not use a generic payment-method refund timeline as the answer "
"to when an undelivered order's refund will be initiated.\n"

In [ ]:
result = support_turn_v2(
    session_id="refund_test_3_retry",
    user_message="My prepaid order was not delivered. When will I get my money back?"
)

print("ANSWER:", result["reply"])
print("SOURCES:", [
    chunk["source"]
    for chunk in result["retrieved_chunks"]
])

In [ ]:
# Test identity verification before order details are shared
test_order_id = str(orders.iloc[0]["order_id"])

result = support_turn_v2(
    session_id="order_security_test_001",
    user_message=f"What is the status of order {test_order_id}?"
)

print("TEST ORDER:", test_order_id)
print("AGENT REPLY:", result["reply"])

In [ ]:
from google.colab import files

uploaded = files.upload()
audio_path = next(iter(uploaded))
print("Uploaded:", audio_path)

In [ ]:
!pip -q install pydub
!apt-get -qq install -y ffmpeg

from pydub import AudioSegment

audio = AudioSegment.from_file(audio_path)
wav_path = "/content/hindi_voice.wav"

audio = audio.set_frame_rate(16000).set_channels(1)
audio.export(wav_path, format="wav")

print("Converted successfully:", wav_path)

In [ ]:
result = voice_support(
    "/content/hindi_voice.wav",
    session_id="hindi_test_001",
    language="hi-IN"
)

In [ ]:
result = support_turn_v2(
    session_id="hindi_retrieval_check",
    user_message="मेरा ऑर्डर कितने दिनों बाद पिकअप होगा?"
)

for i, chunk in enumerate(result["retrieved_chunks"], 1):
    print(f"\n--- Chunk {i} ---")
    print("Source:", chunk["source"])
    print("Score:", chunk["score"])
    print("Text:", chunk["text"][:700])

In [ ]:
for i, text in enumerate(chunk_texts):
    source = chunk_sources[i]
    content = text.lower()

    if (
        "pickup" in content
        or "पिकअप" in content
        or "return collection" in content
        or "collect the return" in content
    ):
        print(f"\n--- Source: {source} ---")
        print(text[:1200])

In [ ]:
result = retrieve_policy_context(
    "reverse pickup booking choose a pickup slot missed pickup rescheduled next business day",
    top_k=5
)

for i, chunk in enumerate(result, 1):
    print(f"\n--- {i}. {chunk['source']} | Score: {chunk['score']:.3f} ---")
    print(chunk["text"][:1000])

In [ ]:
print("Handler exists:", "support_turn_v2" in globals())
print("Retrieval exists:", "retrieve_policy_context" in globals())
print("Embedding model exists:", "embedding_model" in globals())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Test 2: Return pickup
pickup_audio = "/content/return_pickup_query.wav"

tts.synthesize(
    "I have requested a return. When will someone come to pick up my order?",
    voice="Kaveri",
    model="timbre-v2.5",
    language="en-IN",
    output_file=pickup_audio
)

pickup_result = voice_support(
    pickup_audio,
    session_id="pickup_test_001",
    language="en-IN"
)

print("\nCustomer:", pickup_result["customer_text"])
print("Agent:", pickup_result["reply"])
print("Retrieved chunks:", len(pickup_result["retrieved_chunks"]))

In [ ]:
# Test 3: Refund status
refund_audio = "/content/refund_status_query.wav"

tts.synthesize(
    "My return has been picked up. When will I get my refund?",
    voice="Kaveri",
    model="timbre-v2.5",
    language="en-IN",
    output_file=refund_audio
)

refund_result = voice_support(
    refund_audio,
    session_id="refund_test_001",
    language="en-IN"
)

print("\nCustomer:", refund_result["customer_text"])
print("Agent:", refund_result["reply"])
print("Retrieved chunks:", len(refund_result["retrieved_chunks"]))